# Intelligent Neural Network v2

On a montré que le modèle fonctionnait pour de la génération de shakespear, aussi bien qu'un transformer.

On a montré que notre modèle avait une plasticité lui permettant de faire du MultiTasking.

Pour être un SOTA, il lui manque la rapidité. C'est ce que nous allons essayer d'implémenter ici en remplaçant les LSTM par des Mamba/SSM.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import time
import random
import os

## Le MambaBlock

In [45]:
class MultiMambaBlock(nn.Module):
    def __init__(self, num_neurons, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.num_neurons = num_neurons
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.dt_rank = math.ceil(d_model / 16)
        self.d_state = d_state

        self.w_in = nn.Parameter(torch.Tensor(num_neurons, d_model, self.d_inner * 2))
        self.conv1d = nn.Conv1d(
            in_channels=num_neurons * self.d_inner,
            out_channels=num_neurons * self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=num_neurons * self.d_inner,
            padding=d_conv - 1,
        )
        self.w_x = nn.Parameter(torch.Tensor(num_neurons, self.d_inner, self.dt_rank + d_state * 2))
        self.w_dt = nn.Parameter(torch.Tensor(num_neurons, self.dt_rank, self.d_inner))
        self.b_dt = nn.Parameter(torch.Tensor(num_neurons, self.d_inner))

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(num_neurons, self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(num_neurons, self.d_inner))
        self.w_out = nn.Parameter(torch.Tensor(num_neurons, self.d_inner, d_model))

        self.act = nn.SiLU()
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.w_in)
        nn.init.xavier_uniform_(self.w_x)
        nn.init.xavier_uniform_(self.w_dt)
        nn.init.zeros_(self.b_dt)
        nn.init.xavier_uniform_(self.w_out)

    def selective_scan(self, u, dt, A, B, C, D):
        batch_size, n_neurons, seq_len, d_inner = u.shape
        d_state = A.shape[2]
        h = torch.zeros(batch_size, n_neurons, d_inner, d_state, device=u.device)
        ys = []
        for t in range(seq_len):
            dt_t = F.softplus(dt[:, :, t, :])
            dA = torch.exp(dt_t.unsqueeze(-1) * A.unsqueeze(0))
            dB = dt_t.unsqueeze(-1) * B[:, :, t, :].unsqueeze(2)
            u_t = u[:, :, t, :].unsqueeze(-1)
            h = dA * h + dB * u_t
            y_t = torch.sum(h * C[:, :, t, :].unsqueeze(2), dim=-1)
            y_t = y_t + D.unsqueeze(0) * u[:, :, t, :]
            ys.append(y_t)
        return torch.stack(ys, dim=2)

    def forward(self, x):
        batch_size, num_neurons, seq_len, d_model = x.shape
        x_and_res = torch.einsum('bnld,ndk->bnlk', x, self.w_in)
        (x, res) = x_and_res.split(split_size=[self.d_inner, self.d_inner], dim=-1)

        x = x.permute(0, 1, 3, 2).reshape(batch_size, num_neurons * self.d_inner, seq_len)
        x = self.conv1d(x)[:, :, :seq_len]
        x = x.reshape(batch_size, num_neurons, self.d_inner, seq_len).permute(0, 1, 3, 2)
        x = self.act(x)

        x_dbl = torch.einsum('bnli,nip->bnlp', x, self.w_x)
        (dt, B, C) = x_dbl.split(split_size=[self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = torch.einsum('bnlr,nri->bnli', dt, self.w_dt) + self.b_dt.view(1, num_neurons, 1, -1)
        A = -torch.exp(self.A_log.float())
        y = self.selective_scan(x, dt, A, B, C, self.D)

        y = y * self.act(res)
        out = torch.einsum('bnli,nid->bnld', y, self.w_out)
        return out

## INN Parallelisé

In [48]:
class ParallelINN(nn.Module):
    def __init__(self, vocab_size, num_neurons=64, d_model=64, num_layers=4, n_head=4):
        super().__init__()
        self.num_neurons = num_neurons
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model)

        # 1. Projection Entrée Différenciée (Brise la symétrie)
        self.input_proj = nn.Linear(d_model, num_neurons * d_model)

        self.layers = nn.ModuleList([
            INNLayer(num_neurons, d_model, n_head) for _ in range(num_layers)
        ])

        # 2. Projection Sortie Massive (Flux de gradient garanti)
        self.out_proj = nn.Linear(num_neurons * d_model, vocab_size)
        self.norm_f = nn.LayerNorm(d_model)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        x = self.embedding(input_ids)

        # Application Projection Entrée
        x = self.input_proj(x)
        x = x.view(batch_size, seq_len, self.num_neurons, self.d_model)
        x = x.permute(0, 2, 1, 3)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_f(x)

        # Application Sortie Massive
        out_flat = x.permute(0, 2, 1, 3).reshape(batch_size, seq_len, -1)
        logits = self.out_proj(out_flat)
        return logits

class INNLayer(nn.Module):
    def __init__(self, num_neurons, d_model, n_head):
        super().__init__()
        self.mamba = MultiMambaBlock(num_neurons, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.comm_attention = MultiHeadNeuronAttention(num_neurons, d_model, n_head)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x):
        B, N, L, D = x.shape
        res = x
        x = self.mamba(x)
        x = self.norm1(x + res)

        res = x
        x_flat = x.permute(0, 2, 1, 3).reshape(B*L, N, D)
        comm_out = self.comm_attention(x_flat)
        comm_out = comm_out.reshape(B, L, N, D).permute(0, 2, 1, 3)
        x = self.norm2(comm_out + res)

        res = x
        x = self.ffn(x)
        x = self.norm3(x + res)
        return x

class MultiHeadNeuronAttention(nn.Module):
    def __init__(self, num_neurons, d_model, n_head):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_head, batch_first=True)
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        return attn_out

## Entrainement

In [17]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-11-22 23:56:02--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2025-11-22 23:56:02 (23.1 MB/s) - ‘input.txt.2’ saved [1115394/1115394]



In [19]:
# --- Config ---
BATCH_SIZE = 32
SEQ_LEN = 128
LEARNING_RATE = 1e-3
EPOCHS = 10
NUM_NEURONS = 64
D_MODEL = 64
NUM_LAYERS = 4
PRINT_EVERY = 50

def load_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for i, ch in enumerate(chars)}
    return text, chars, char2idx, idx2char, vocab_size

def get_batch(text, char2idx, batch_size, seq_len, vocab_size):
    input_seqs = []
    target_seqs = []
    for _ in range(batch_size):
        start_idx = random.randint(0, len(text) - seq_len - 1)
        chunk = text[start_idx : start_idx + seq_len + 1]
        input_indices = [char2idx[c] for c in chunk[:-1]]
        target_indices = [char2idx[c] for c in chunk[1:]]
        input_seqs.append(torch.tensor(input_indices, dtype=torch.long))
        target_seqs.append(torch.tensor(target_indices, dtype=torch.long))
    return torch.stack(input_seqs), torch.stack(target_seqs)

def generate_text(model, start_str, char2idx, idx2char, vocab_size, length=100, temperature=0.8):
    model.eval()
    device = next(model.parameters()).device
    input_indices = [char2idx[c] for c in start_str]
    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)
    generated_str = start_str

    with torch.no_grad():
        for _ in range(length):
            if input_tensor.size(1) > SEQ_LEN:
                context = input_tensor[:, -SEQ_LEN:]
            else:
                context = input_tensor

            logits = model(context)
            last_logits = logits[:, -1, :]
            probs = torch.softmax(last_logits / temperature, dim=-1)
            next_idx = torch.multinomial(probs, 1).item()

            generated_str += idx2char[next_idx]
            next_tensor = torch.tensor([[next_idx]], device=device)
            input_tensor = torch.cat([input_tensor, next_tensor], dim=1)

    model.train()
    return generated_str

def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Utilisation du device : {device}")

    # Recherche du fichier input.txt
    paths = ['input.txt', '../INNv1/test/input.txt', 'INNv1/test/input.txt']
    dataset_file = None
    for p in paths:
        if os.path.exists(p):
            dataset_file = p
            break

    if not dataset_file:
        # Fallback création dummy
        print("Info: input.txt non trouvé, création d'un dummy pour le test.")
        dataset_file = 'dummy_input.txt'
        with open(dataset_file, 'w') as f:
            f.write("To be or not to be " * 1000)

    text, chars, char2idx, idx2char, vocab_size = load_data(dataset_file)
    print(f"Dataset: {dataset_file} ({len(text)} chars)")

    model = ParallelINN(vocab_size, NUM_NEURONS, D_MODEL, NUM_LAYERS).to(device)
    print(f"Modèle Parallel INN: {sum(p.numel() for p in model.parameters())} params")

    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    num_batches = 200

    for epoch in range(EPOCHS):
        start_time = time.time()
        total_loss = 0

        for i in range(num_batches):
            inputs, targets = get_batch(text, char2idx, BATCH_SIZE, SEQ_LEN, vocab_size)
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

            if i % PRINT_EVERY == 0:
                print(f"Batch {i}, Loss: {loss.item():.4f}")

        duration = time.time() - start_time
        avg_loss = total_loss / num_batches
        print(f"=== Epoch {epoch+1} en {duration:.2f}s | Loss: {avg_loss:.4f} ===")
        print(generate_text(model, "The ", char2idx, idx2char, vocab_size))
        print("-" * 30)

if __name__ == "__main__":
    train()



Utilisation du device : cuda
Dataset: input.txt (1115394 chars)
Modèle Parallel INN: 8593921 params
Batch 0, Loss: 4.3300
Batch 50, Loss: 2.1697
Batch 100, Loss: 1.8410
Batch 150, Loss: 1.6719
=== Epoch 1 en 264.92s | Loss: 2.0014 ===
The away, here made smite,
But hear your forth Rome?
That the curself, in the honderning too, him, for A
------------------------------
Batch 0, Loss: 1.5736
Batch 50, Loss: 1.5779
Batch 100, Loss: 1.4790
Batch 150, Loss: 1.4809
=== Epoch 2 en 264.86s | Loss: 1.4989 ===
The sounded hill't, since's their strike.

CORIOLANUS:
Contenty.

Second Servant:
Such she had my follow
------------------------------
Batch 0, Loss: 1.4483
Batch 50, Loss: 1.3971
Batch 100, Loss: 1.4725
Batch 150, Loss: 1.3927
=== Epoch 3 en 264.86s | Loss: 1.4067 ===
The queen and your king.

First
With mightable and tribute?

KING RICHARD III:
We will be his or in this
------------------------------
Batch 0, Loss: 1.3386
Batch 50, Loss: 1.3370
Batch 100, Loss: 1.3334
Batch 150, Loss: 1

# Comparaison à Mamba Pure

In [20]:
# On réutilise la fonction de scan exact de votre implémentation pour une comparaison équitable
def selective_scan_ref(u, dt, A, B, C, D):
    """
    Standard Selective Scan (Single Channel / Monolithic).
    u: (B, L, d_inner)
    dt: (B, L, d_inner)
    A: (d_inner, d_state)
    B: (B, L, d_state)
    C: (B, L, d_state)
    D: (d_inner)
    """
    batch_size, seq_len, d_inner = u.shape
    d_state = A.shape[1]

    h = torch.zeros(batch_size, d_inner, d_state, device=u.device)
    ys = []

    # Scan séquentiel (pour la baseline, la vitesse pure n'est pas critique, la justesse mathématique l'est)
    # Note: Une implémentation CUDA optimisée serait plus rapide mais cette version python suffit pour la validation
    for t in range(seq_len):
        dt_t = F.softplus(dt[:, t, :]) # (B, d_inner)

        # Discretization
        dA = torch.exp(dt_t.unsqueeze(-1) * A.unsqueeze(0)) # (B, d_inner, d_state)
        dB = dt_t.unsqueeze(-1) * B[:, t, :].unsqueeze(1)   # (B, d_inner, d_state)

        u_t = u[:, t, :].unsqueeze(-1) # (B, d_inner, 1)

        # State Update
        h = dA * h + dB * u_t

        # Output
        y_t = torch.sum(h * C[:, t, :].unsqueeze(1), dim=-1) # (B, d_inner)
        y_t = y_t + D.unsqueeze(0) * u[:, t, :]

        ys.append(y_t)

    return torch.stack(ys, dim=1) # (B, L, d_inner)


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = int(expand * d_model)
        self.dt_rank = math.ceil(d_model / 16)
        self.d_state = d_state

        # 1. In Projection
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)

        # 2. Conv1d
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )

        # 3. SSM Parameters
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        # S4D initialization for A
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # 4. Out Projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.act = nn.SiLU()

    def forward(self, x):
        # x: (B, L, D)
        batch_size, seq_len, d_model = x.shape

        # 1. Proj
        x_and_res = self.in_proj(x)  # (B, L, 2*d_inner)
        (x, res) = x_and_res.split(split_size=[self.d_inner, self.d_inner], dim=-1)

        x = x.permute(0, 2, 1) # (B, d_inner, L)
        x = self.conv1d(x)[:, :, :seq_len]
        x = x.permute(0, 2, 1) # (B, L, d_inner)

        x = self.act(x)

        # 2. SSM
        x_dbl = self.x_proj(x) # (B, L, dt_rank + 2*d_state)
        (dt, B, C) = x_dbl.split(split_size=[self.dt_rank, self.d_state, self.d_state], dim=-1)

        dt = self.dt_proj(dt) # (B, L, d_inner)

        A = -torch.exp(self.A_log.float()) # (d_inner, d_state)

        y = selective_scan_ref(x, dt, A, B, C, self.D)

        # 3. Output
        y = y * self.act(res)
        out = self.out_proj(y)

        return out

class MambaPure(nn.Module):
    """
    Architecture Mamba Pure : Stack de MambaBlocks.
    Conçue pour matcher le nombre de paramètres de l'INNv2 (~8.6M).
    """
    def __init__(
        self,
        vocab_size,
        d_model=384, # Ajusté pour ~8.6M params (à vérifier au runtime)
        n_layers=4,
        d_state=16,
        d_conv=4,
        expand=2,
        max_seq_len=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            nn.Sequential(
                MambaBlock(d_model, d_state, d_conv, expand),
                nn.LayerNorm(d_model)
            )
            for _ in range(n_layers)
        ])

        self.norm_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Tie weights
        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids):
        # input_ids: (B, L)
        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x) + x # Residual connection classique

        x = self.norm_f(x)
        logits = self.lm_head(x)

        return logits



In [22]:
# ============================================
# CONFIGURATION - IDENTIQUE À TRAIN_PARALLEL.PY
# ============================================
# Params INNv2:
# BATCH_SIZE = 32
# SEQ_LEN = 128
# LEARNING_RATE = 1e-3
# EPOCHS = 10
# NUM_LAYERS = 4
# NUM_NEURONS = 64, D_MODEL = 64 -> 8.6M params total

CONFIG = {
    'batch_size': 32,
    'seq_len': 128,
    'learning_rate': 1e-3,
    'epochs': 10,
    'n_layers': 4,

    # Tuning pour matcher 8.6M params :
    # INNv2 params ~ 8.6M
    # MambaPure avec d_model=480 -> ~8.5M params
    'd_model': 480,

    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'log_file': 'results/mamba_pure_results.json'
}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def load_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chars = sorted(list(set(text)))
    vocab_size = len(chars)
    char2idx = {ch: i for i, ch in enumerate(chars)}
    idx2char = {i: ch for i, ch in enumerate(chars)}
    return text, chars, char2idx, idx2char, vocab_size

def get_batch(text, char2idx, batch_size, seq_len, vocab_size):
    input_seqs = []
    target_seqs = []
    for _ in range(batch_size):
        start_idx = random.randint(0, len(text) - seq_len - 1)
        chunk = text[start_idx : start_idx + seq_len + 1]
        input_indices = [char2idx[c] for c in chunk[:-1]]
        target_indices = [char2idx[c] for c in chunk[1:]]
        input_seqs.append(torch.tensor(input_indices, dtype=torch.long))
        target_seqs.append(torch.tensor(target_indices, dtype=torch.long))
    return torch.stack(input_seqs), torch.stack(target_seqs)

def main():
    print("=== TRAINING MAMBA PURE BASELINE ===")
    device = torch.device(CONFIG['device'])
    torch.manual_seed(42)

    # 1. Data
    paths = ['test/input.txt', 'input.txt', 'INNv1/test/input.txt']
    dataset_file = None
    for p in paths:
        if os.path.exists(p):
            dataset_file = p
            break
    if not dataset_file:
        print("Erreur: input.txt non trouvé")
        return

    text, chars, char2idx, idx2char, vocab_size = load_data(dataset_file)
    print(f"Vocab size: {vocab_size}")

    # 2. Model
    model = MambaPure(
        vocab_size=vocab_size,
        d_model=CONFIG['d_model'],
        n_layers=CONFIG['n_layers']
    ).to(device)

    num_params = count_parameters(model)
    print(f"Mamba Pure Params: {num_params:,} ({num_params/1e6:.2f}M)")
    print("Target: ~8.60M")

    # 3. Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=CONFIG['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    # 4. Training Loop
    history = []
    num_batches = 200 # Comme dans train_parallel.py

    for epoch in range(CONFIG['epochs']):
        start_time = time.time()
        total_loss = 0
        model.train()

        for i in range(num_batches):
            inputs, targets = get_batch(text, char2idx, CONFIG['batch_size'], CONFIG['seq_len'], vocab_size)
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

            if i % 50 == 0:
                print(f"Epoch {epoch+1} Batch {i} Loss {loss.item():.4f}")

        avg_loss = total_loss / num_batches
        epoch_time = time.time() - start_time

        print(f"=== Epoch {epoch+1} Done | Loss: {avg_loss:.4f} | Time: {epoch_time:.2f}s ===")

        history.append({
            'epoch': epoch + 1,
            'val_loss': avg_loss, # On utilise train loss comme val loss proxy ici (dataset simple)
            'time': epoch_time,
            'val_ppl': torch.exp(torch.tensor(avg_loss)).item()
        })

        # Save checkpoints
        if epoch == CONFIG['epochs'] - 1:
            torch.save(model.state_dict(), 'checkpoints/mamba_pure_final.pt')

    # Save results
    os.makedirs('results', exist_ok=True)
    with open(CONFIG['log_file'], 'w') as f:
        json.dump(history, f, indent=2)
    print(f"Results saved to {CONFIG['log_file']}")

if __name__ == "__main__":
    main()



=== TRAINING MAMBA PURE BASELINE ===
Vocab size: 65
Mamba Pure Params: 6,007,200 (6.01M)
Target: ~8.60M
Epoch 1 Batch 0 Loss 211.8517
Epoch 1 Batch 50 Loss 4.6231
Epoch 1 Batch 100 Loss 3.1257
Epoch 1 Batch 150 Loss 2.5246
=== Epoch 1 Done | Loss: 6.3813 | Time: 94.46s ===
Epoch 2 Batch 0 Loss 2.3092
Epoch 2 Batch 50 Loss 2.2444
Epoch 2 Batch 100 Loss 1.9900
Epoch 2 Batch 150 Loss 1.8233
=== Epoch 2 Done | Loss: 2.0630 | Time: 94.67s ===
Epoch 3 Batch 0 Loss 1.8699
Epoch 3 Batch 50 Loss 1.8436
Epoch 3 Batch 100 Loss 1.7086
Epoch 3 Batch 150 Loss 1.7978
=== Epoch 3 Done | Loss: 1.7969 | Time: 94.37s ===
Epoch 4 Batch 0 Loss 1.6120
Epoch 4 Batch 50 Loss 1.5877
Epoch 4 Batch 100 Loss 1.7336
Epoch 4 Batch 150 Loss 1.7009
=== Epoch 4 Done | Loss: 1.6592 | Time: 94.18s ===
Epoch 5 Batch 0 Loss 1.5694
Epoch 5 Batch 50 Loss 1.6223
Epoch 5 Batch 100 Loss 1.6381
Epoch 5 Batch 150 Loss 1.4628
=== Epoch 5 Done | Loss: 1.5735 | Time: 94.54s ===
Epoch 6 Batch 0 Loss 1.4351
Epoch 6 Batch 50 Loss 1.51

RuntimeError: Parent directory checkpoints does not exist.

# Test INNv2 sur WikiText-2

Chargement du dataset

In [34]:
# --- CELLULE 2 : DATA LOADING (WikiText-2) ---
import os
import torch
from collections import Counter
import requests

# 1. Download
def download_wikitext():
    urls = {
        "train.txt": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt",
        "valid.txt": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/valid.txt",
        "test.txt": "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/test.txt"
    }
    os.makedirs("data/wikitext-2", exist_ok=True)
    for name, url in urls.items():
        path = f"data/wikitext-2/{name}"
        if not os.path.exists(path):
            print(f"Downloading {name}...")
            r = requests.get(url)
            with open(path, 'w', encoding='utf-8') as f:
                f.write(r.text)
    print("Data ready.")

download_wikitext()

# 2. Tokenizer (Word-Level)
class Dictionary:
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []
        self.counter = Counter()
        self.total = 0

    def add_word(self, word):
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
        token_id = self.word2idx[word]
        self.counter[token_id] += 1
        self.total += 1
        return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)

class Corpus:
    def __init__(self, path):
        self.dictionary = Dictionary()
        self.train = self.tokenize(os.path.join(path, 'train.txt'), build_vocab=True)
        self.valid = self.tokenize(os.path.join(path, 'valid.txt'))
        self.test = self.tokenize(os.path.join(path, 'test.txt'))

    def tokenize(self, path, build_vocab=False):
        assert os.path.exists(path)
        with open(path, 'r', encoding="utf8") as f:
            tokens = 0
            for line in f:
                words = line.split() + ['<eos>']
                tokens += len(words)
                if build_vocab:
                    for word in words:
                        self.dictionary.add_word(word)

        with open(path, 'r', encoding="utf8") as f:
            ids = torch.LongTensor(tokens)
            token = 0
            for line in f:
                words = line.split() + ['<eos>']
                for word in words:
                    if word in self.dictionary.word2idx:
                        ids[token] = self.dictionary.word2idx[word]
                    token += 1
        return ids

def batchify(data, bsz, device):
    nbatch = data.size(0) // bsz
    data = data.narrow(0, 0, nbatch * bsz)
    data = data.view(bsz, -1).t().contiguous()
    return data.to(device)

# Load Corpus
print("Building Corpus...")
corpus = Corpus('data/wikitext-2')
vocab_size = len(corpus.dictionary)
print(f"Vocab Size: {vocab_size}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 20
eval_batch_size = 10

train_data = batchify(corpus.train, batch_size, device)
val_data = batchify(corpus.valid, eval_batch_size, device)
test_data = batchify(corpus.test, eval_batch_size, device)
print("Data Loaded.")

Data ready.
Building Corpus...
Vocab Size: 33278
Data Loaded.


Entrainement

In [ ]:
# --- CELLULE 3 (CORRIGÉE) : TRAIN INNv2 AVEC WARMUP ---
import time
import math
import torch.optim as optim

# Hyperparams Ajustés
SEQ_LEN = 64
LEARNING_RATE = 5e-4  # Réduit pour stabilité
EPOCHS = 15
NUM_NEURONS = 16
D_MODEL = 256
NUM_LAYERS = 4
WARMUP_STEPS = 1000   # Important !

model = ParallelINN(
    vocab_size=vocab_size,
    num_neurons=NUM_NEURONS,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS
).to(device)

# Initialisation spécifique des embeddings
nn.init.normal_(model.embedding.weight, mean=0.0, std=0.02)
nn.init.normal_(model.out_proj.weight, mean=0.0, std=0.02)
nn.init.zeros_(model.out_proj.bias)

print(f"INNv2 Params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1)
criterion = nn.CrossEntropyLoss()

# Scheduler avec Warmup -> Cosine Decay (Standard moderne)
total_steps = (train_data.size(0) // SEQ_LEN) * EPOCHS
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=total_steps,
    pct_start=0.1  # 10% de warmup
)

def get_batch(source, i):
    seq_len = min(SEQ_LEN, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].view(-1)
    return data, target

def evaluate(data_source):
    model.eval()
    total_loss = 0.
    with torch.no_grad():
        for i in range(0, data_source.size(0) - 1, SEQ_LEN):
            data, targets = get_batch(data_source, i)
            output = model(data.t())
            output_flat = output.view(-1, vocab_size)
            total_loss += len(data) * criterion(output_flat, targets).item()
    return total_loss / (len(data_source) - 1)

print("Starting Training (v2 - Optimized)...")
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    model.train()
    total_loss = 0.

    for batch, i in enumerate(range(0, train_data.size(0) - 1, SEQ_LEN)):
        data, targets = get_batch(train_data, i)
        data = data.t()

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output.reshape(-1, vocab_size), targets)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if batch % 200 == 0 and batch > 0:
            cur_loss = total_loss / 200
            print(f"| epoch {epoch} | batch {batch} | loss {cur_loss:.2f} | ppl {math.exp(cur_loss):.2f} | lr {scheduler.get_last_lr()[0]:.2e}")
            total_loss = 0

    val_loss = evaluate(val_data)
    val_ppl = math.exp(val_loss)
    print('-' * 60)
    print(f"| End of epoch {epoch} | time: {time.time() - epoch_start_time:.2f}s | "
          f"valid loss {val_loss:.2f} | valid ppl {val_ppl:.2f}")
    print('-' * 60)

INNv2 Params: 177,090,046
Starting Training (v2 - Optimized)...
| epoch 1 | batch 200 | loss 7.61 | ppl 2012.69 | lr 2.80e-05
| epoch 1 | batch 400 | loss 7.20 | ppl 1342.53 | lr 5.11e-05
| epoch 1 | batch 600 | loss 7.18 | ppl 1314.40 | lr 8.80e-05
